In [ ]:
import json
from collections import defaultdict
from pathlib import Path

def validate_manifest_pairs(file_path: str):
    """
    Goes over every line in the manifest and checks if each prompt hash
    has exactly two unique samples: one ending in '_A' and one ending in '_B'.
    """
    file_path = Path(file_path)
    if not file_path.exists():
        print(f"❌ Error: File not found at {file_path}")
        return

    # Dictionary to hold the samples found for each prompt hash
    # Structure: { 'prompt_hash': set of ['_A', '_B'] }
    prompt_hash_map = defaultdict(set)
    total_lines = 0
    errors_found = False

    print(f"🔬 Starting validation check on {file_path.name}...")

    # 1. First Pass: Aggregate Samples
    with open(file_path, 'r') as f:
        for line_number, line in enumerate(f, 1):
            total_lines += 1
            try:
                data = json.loads(line)
                prompt_hash = data.get('prompt_hash')
                sample_id = data.get('sample_id')

                if not prompt_hash or not sample_id:
                    print(f"🚨 Line {line_number}: Missing 'prompt_hash' or 'sample_id'. Data: {line.strip()}")
                    errors_found = True
                    continue

                # The sample key is the suffix (_A, _B)
                if sample_id.endswith('_A'):
                    sample_key = '_A'
                elif sample_id.endswith('_B'):
                    sample_key = '_B'
                else:
                    print(f"⚠️ Line {line_number}: Unexpected sample ID suffix. Expected '_A' or '_B'. ID: {sample_id}")
                    errors_found = True
                    continue

                # Check for direct duplicates (same hash and same key, e.g., two '_A's)
                if sample_key in prompt_hash_map[prompt_hash]:
                    print(f"❌ Line {line_number}: DUPLICATE SAMPLE FOUND! Prompt Hash: {prompt_hash}, Sample: {sample_id}")
                    errors_found = True
                
                prompt_hash_map[prompt_hash].add(sample_key)

            except json.JSONDecodeError:
                print(f"🚨 Line {line_number}: Invalid JSON format. Skipping line: {line.strip()}")
                errors_found = True

    # 2. Second Pass: Verify Pairs
    prompts_with_errors = []
    
    for prompt_hash, suffixes in prompt_hash_map.items():
        if suffixes != {'_A', '_B'}:
            prompts_with_errors.append({
                'hash': prompt_hash,
                'count': len(suffixes),
                'found': sorted(list(suffixes)),
                'prompt': next((data['prompt'] for data in [json.loads(line) for line in file_path.open() if json.loads(line).get('prompt_hash') == prompt_hash]), "N/A")
            })

    # 3. Final Summary
    total_prompts = len(prompt_hash_map)
    total_samples = total_lines

    if not errors_found and not prompts_with_errors:
        print("\n✅ VALIDATION SUCCESSFUL! 🎉")
        print(f"Total Prompts (Hashes): {total_prompts}")
        print(f"Total Samples (Lines): {total_samples}")
        print("Every prompt has exactly one '_A' and one '_B' sample.")
    else:
        print("\n❌ VALIDATION FAILED. Errors detected in manifest integrity.")
        
        # Report missing/incomplete pairs
        if prompts_with_errors:
            print(f"\n--- INCOMPLETE OR OVER-GENERATED PROMPTS ({len(prompts_with_errors)} errors) ---")
            for error in prompts_with_errors:
                status = "Missing Sample" if error['count'] < 2 else "Unexpected Sample"
                print(f"[{status}] Hash: {error['hash']} | Prompt: {error['prompt'][:70]}... | Found: {error['found']} (Count: {error['count']})")
        
        print("\n--- SUMMARY ---")
        print(f"Total Lines Processed: {total_lines}")
        print(f"Total Unique Prompts: {total_prompts}")
        print("Action required: Review lines flagged with '❌ DUPLICATE SAMPLE FOUND' or fix the prompts listed above.")


# --- Execution ---
# IMPORTANT: Replace 'generation_manifest.jsonl' with the actual path if the file is not in the script's directory.
validate_manifest_pairs("ratings_interface/rlhf_generation_data/generation_manifest.jsonl")

In [4]:
import json
from pathlib import Path
from typing import List, Dict, Any, Union
import re

def extract_age(prompt: str) -> int:
    """
    Extracts the numeric age from the beginning of the prompt string.
    e.g., "A 45 years old..." -> 45
    e.g., "A 5 years old..." -> 5
    """
    # Regex to find the first number that follows "A "
    match = re.search(r"^A (\d+) years old", prompt)
    if match:
        return int(match.group(1))
    
    # Fallback for unexpected formats, though rare
    return 999 

def deduplicate_and_sort_manifest(input_file: str):
    """
    1. Deduplicates the JSONL file by reading backwards (preserving the LATEST entry).
    2. Sorts the resulting unique entries numerically by the extracted age,
       and then alphabetically by the full prompt text for stable ordering.
    """
    input_path = Path(input_file)
    # New descriptive output file name
    output_path = input_path.with_name(f"{input_path.stem}_clean_ordered_by_age.jsonl") 
    
    if not input_path.exists():
        print(f"❌ Error: File not found at {input_file}")
        return

    # --- 1. Deduplication Pass (Reverse Read) ---
    print(f"🔬 Reading all lines from {input_file} for reverse deduplication...")
    with open(input_path, 'r') as f:
        all_lines = f.readlines()
        
    unique_samples_found = set()
    unique_data_objects: List[Dict[str, Any]] = []
    duplicate_count = 0
    
    # Iterate backwards to keep the latest entries
    for line in reversed(all_lines):
        try:
            data = json.loads(line)
            sample_id = data.get('sample_id')

            if not sample_id:
                continue

            # Preserve the LATEST version
            if sample_id not in unique_samples_found:
                unique_samples_found.add(sample_id)
                unique_data_objects.append(data)
            else:
                duplicate_count += 1
                
        except json.JSONDecodeError:
            print(f"⚠️ Warning: Skipping line due to invalid JSON: {line.strip()[:50]}...")
            continue
    
    # Restore original chronological order of unique entries before sorting
    unique_data_objects.reverse()
    print(f"✅ Deduplication complete. Total duplicates removed: {duplicate_count}")
    
    # --- 2. Sorting Pass (Numerically by Age) ---
    print("➡️ Sorting data numerically by extracted age...")
    
    # Sort using a tuple of (extracted_age, prompt_text) for stable sorting
    sorted_data_objects = sorted(
        unique_data_objects, 
        key=lambda x: (extract_age(x['prompt']), x['prompt'])
    )
    
    # --- 3. Write the final clean, ordered file ---
    print(f"✍️ Writing clean, age-ordered data to {output_path.name}...")
    with open(output_path, 'w') as outfile:
        for data in sorted_data_objects:
            outfile.write(json.dumps(data) + '\n')
            
    print(f"\n🎉 Process Complete!")
    print(f"New clean and age-ordered manifest saved to: **{output_path.name}**")
    print(f"Final Total Unique Samples: {len(sorted_data_objects)}")

# --- Execution ---
# Run the function on your duplicated file
# If you are running this in the same environment as before, the path should be correct.
deduplicate_and_sort_manifest("ratings_interface/rlhf_generation_data/generation_manifest.jsonl")

🔬 Reading all lines from ratings_interface/rlhf_generation_data/generation_manifest.jsonl for reverse deduplication...
✅ Deduplication complete. Total duplicates removed: 17
➡️ Sorting data numerically by extracted age...
✍️ Writing clean, age-ordered data to generation_manifest_clean_ordered_by_age.jsonl...

🎉 Process Complete!
New clean and age-ordered manifest saved to: **generation_manifest_clean_ordered_by_age.jsonl**
Final Total Unique Samples: 800
